# Signal study

The tear sheet of one signal under the evaluation protocol. Set `SIGNAL` and run all.

The protocol, in short:

- **Development and holdout.** The sessions before 2025-10-01 are development, split into
  four contiguous folds of about a year (`research.eval_sessions`). The sessions from
  2025-10-01 are the holdout. A signal is judged on development and meets the holdout
  once, when it is final. The holdout is spent for the eleven original signals of
  `research.signals`, because their full-sample IC was read before the holdout existed,
  and for `short_ratio` and `days_to_cover`, which met it once. `short_change` died on
  development, so its holdout is unread.
- **Purge.** A label of horizon h ends h sessions later. A period uses a session only
  when its label ends inside the same period.
- **The t-statistic.** Daily ICs of a 21-session label overlap, so `t_naive` is about
  four times too high. Quote `t_nw`, the Newey-West t. Many signals are tested, so a t
  under 3 is weak evidence.
- **Two targets.** `return` is the forward return. `residual` is the forward residual
  of the style and industry model, so a residual IC is a prediction beyond the known
  factors.


In [ ]:
import numpy as np
import plotly.express as px
import polars as pl

from sdp import dal

wh = dal.warehouse()   # A read-only connection to the dbt warehouse.

SIGNAL = "momentum_12_1"
HORIZON = 21           # 1, 5 or 21 sessions

signals = [r[0] for r in wh.sql("select distinct signal from research.signal_ic "
                                "order by 1").fetchall()]
assert SIGNAL in signals, f"{SIGNAL} is not one of {signals}"

## 1. The IC on development

One row per target and horizon. `folds_same_sign` counts the four folds whose mean IC
has the sign of the whole.


In [ ]:
summary = wh.sql(f"""
    select target, horizon, n_days, mean_ic, ic_std, t_naive, t_nw, ic_autocorr_lag1,
           folds_same_sign
    from research.signal_ic_summary
    where signal = '{SIGNAL}' and period = 'development'
    order by target, horizon
""").pl()
summary.with_columns(pl.col("mean_ic", "ic_std").round(4),
                     pl.col("t_naive", "t_nw", "ic_autocorr_lag1").round(2))

In [ ]:
folds = wh.sql(f"""
    select target, period, mean_ic, t_nw
    from research.signal_ic_summary
    where signal = '{SIGNAL}' and horizon = {HORIZON} and period like 'fold%'
    order by target, period
""").pl()
fig = px.bar(folds.to_pandas(), x="period", y="mean_ic", color="target", barmode="group",
             hover_data=["t_nw"], title=f"{SIGNAL}: mean IC by fold, horizon {HORIZON}")
fig.update_layout(height=380, xaxis_title="", yaxis_title="mean IC")
fig.show()

## 2. The daily IC

The daily Spearman IC against the forward return of `HORIZON` sessions, and its mean over
63 sessions. The shaded area is the holdout.


In [ ]:
ic = wh.sql(f"""
    select date, target, ic from research.signal_ic
    where signal = '{SIGNAL}' and horizon = {HORIZON}
    order by date
""").pl().with_columns(ic_63=pl.col("ic").rolling_mean(63).over("target"))
holdout = wh.sql("select min(date) from research.eval_sessions "
                 "where period = 'holdout'").fetchone()[0]
fig = px.line(ic.to_pandas(), x="date", y="ic_63", color="target",
              title=f"{SIGNAL}: IC, 63-session mean, horizon {HORIZON}")
fig.add_vrect(x0=holdout, x1=ic["date"].max(), fillcolor="#9aa0a6", opacity=0.15,
              line_width=0, annotation_text="holdout")
fig.add_hline(y=0, line_color="#666")
fig.update_layout(height=400, yaxis_title="IC")
fig.show()

## 3. The shape

The mean forward return of each quintile of the signal over the full sample, holdout
included. A monotone shape is a better sign than a result that lives in one tail.


In [ ]:
q = wh.sql(f"""
    select quintile, n_obs, mean_fwd_1, mean_fwd_5, mean_fwd_21
    from research.signal_quintiles where signal = '{SIGNAL}' order by quintile
""").pl()
fig = px.bar(q.to_pandas(), x="quintile", y=f"mean_fwd_{HORIZON}",
             title=f"{SIGNAL}: mean {HORIZON}-session forward return by quintile")
fig.update_layout(height=360, yaxis_tickformat=".2%")
fig.show()

## 4. The long-short after cost

The top quintile less the bottom quintile, equal weight, rebalanced each session. For a
signal with a negative IC, the bet is the reverse. The cost is half the model spread of
`research.spreads` on each name that enters or leaves a leg. `capacity` is the size of
one leg at which each equal-weight position is 5% of the daily dollar volume of the
tenth-percentile name of the leg.


In [ ]:
ls = wh.sql(f"""
    select date, gross, cost, net, turnover, capacity
    from research.signal_costs where signal = '{SIGNAL}' order by date
""").pl()
curve = ls.select("date", *[pl.col(c).fill_null(0).cum_sum().alias(c)
                            for c in ("gross", "net")])
fig = px.line(curve.unpivot(index="date", variable_name="series", value_name="cum").to_pandas(),
              x="date", y="cum", color="series",
              title=f"{SIGNAL}: cumulative long-short return, gross and net")
fig.update_layout(height=400, yaxis_tickformat=".0%", yaxis_title="sum of daily returns")
fig.show()


def sharpe(x):
    x = x.drop_nulls()
    return float(x.mean() / x.std() * np.sqrt(252))


pl.DataFrame([{
    "sharpe_gross": round(sharpe(ls["gross"]), 2),
    "sharpe_net": round(sharpe(ls["net"]), 2),
    "cost_bps_a_day": round(float(ls["cost"].mean() * 1e4), 2),
    "turnover_a_day": round(float(ls["turnover"].mean()), 3),
    "capacity_median_musd": round(float(ls["capacity"].median() / 1e6), 0),
}])

## 5. Against the other signals

The mean rank correlation of the signal with each other signal. A new signal that
correlates highly with an old one adds little.


In [ ]:
wh.sql(f"""
    select case when signal_a = '{SIGNAL}' then signal_b else signal_a end as other,
           round(rank_corr, 3) as rank_corr
    from research.signal_correlation
    where '{SIGNAL}' in (signal_a, signal_b) and signal_a <> signal_b
    order by abs(rank_corr) desc
""").pl()

## 6. The scoreboard

Every signal, development only, horizon `HORIZON`, both targets. The last column marks
a `t_nw` above 3.


In [ ]:
wh.sql(f"""
    select signal, target, round(mean_ic, 4) as mean_ic, round(t_nw, 2) as t_nw,
           folds_same_sign, abs(t_nw) >= 3 as strong
    from research.signal_ic_summary
    where period = 'development' and horizon = {HORIZON}
    order by target, abs(t_nw) desc
""").pl()

## 7. Names that lose their label

A label needs a price `HORIZON` sessions later. A name that delists, is acquired or
stops trading before then has no label, and the IC drops it. The vendor gives no
delisting return, so the IC never sees the last move of such a name. The cells below
measure how often this happens and whether the lost names sit in one tail of the signal,
which would bias the long-short.

A name whose label is missing only because the data ends is not counted.


In [ ]:
lost = wh.sql(f"""
    with last_session as (
        select date from research.eval_sessions order by date desc limit 1 offset {HORIZON}
    )
    select date_part('year', date) as year,
           count(*) as rows,
           count(*) filter (where fwd_ret_{HORIZON} is null) as lost,
           round(100.0 * count(*) filter (where fwd_ret_{HORIZON} is null) / count(*), 3)
               as lost_pct,
           round(avg(quintile) filter (where fwd_ret_{HORIZON} is null), 2)
               as mean_quintile_of_lost
    from research.signal_panel
    where signal = '{SIGNAL}' and date <= (select date from last_session)
    group by 1 order by 1
""").pl()
lost

`mean_quintile_of_lost` is 3 when the lost names are spread evenly over the signal. A
value far from 3 means the signal predicts which names disappear, and the missing last
return then flatters or hurts the long-short. The next cell shows the full distribution.


In [ ]:
dist = wh.sql(f"""
    with last_session as (
        select date from research.eval_sessions order by date desc limit 1 offset {HORIZON}
    )
    select quintile, count(*) as lost
    from research.signal_panel
    where signal = '{SIGNAL}' and fwd_ret_{HORIZON} is null
      and date <= (select date from last_session)
    group by 1 order by 1
""").pl()
fig = px.bar(dist.to_pandas(), x="quintile", y="lost",
             title=f"{SIGNAL}: names that lose their {HORIZON}-session label, by quintile")
fig.update_layout(height=340)
fig.show()

## Notes

Measured on 2026-09-27 for `momentum_12_1`, horizon 21: about 0.4% of the in-universe
rows lose their label each year, and the lost names sit high in the signal (a mean
quintile of 3.5 to 4.1). A likely reading, not yet checked name by name: most lost names
were acquired, rose before the deal, and miss only the small last move to the deal
price. The few that failed sit in the short leg and miss a large loss, which a short
would have earned. If that reading is right, the missing returns make momentum look
slightly worse than it is, not better. The size needs the delisting returns, which the
vendor does not give.